# Session 1 — First Contact

### AI Image Studio · Session 1 of 8

Today has exactly one goal: **get one picture on your screen that you described in words.**

We are not going to explain how it all works today. We're going to make it work first, and
then spend the next seven sessions taking it apart and understanding every piece. That
order is deliberate — it's much easier to understand an explanation of something you have
already seen happen.

## TODAY:

- **Part A:** check that your setup is actually working
- **Part B:** say hello to a model
- **Part C:** make your first image
- **Part D:** save it, then break it on purpose

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🛑 Before you run anything</h3>
<p style="color:#900; margin-bottom:0;">
Look at the <b>top right</b> of this notebook. Does it say <b>Select Kernel</b>?<br/><br/>
If it does, click it → <b>Python Environments...</b> → pick the one that says <code>.venv</code>
and has a star next to it.<br/><br/>
You have to do this <b>for every notebook you open</b>. Almost every "but I installed it!"
problem in this workshop is this, and only this.
</p>
</div>

### How a notebook works

A notebook is a list of **cells**. Some cells are text (like this one), some are code.

Click on a code cell and press **Shift + Enter** to run it. The result appears underneath.

Two rules that will save you a lot of pain:

1. **Run every cell, from the top, in order.** Cells share memory. A cell halfway down
   can only use a variable if the cell that created it has already been run.
2. **If something stops making sense, restart the kernel and run everything again.**
   The restart button is at the top of the notebook. This fixes a surprising amount.

## PART A: Is your setup actually working?

In [ ]:
# imports

import os
import base64
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain.chat_models import init_chat_model

# If this cell gives you a red error, do not continue - raise your hand.
# It means either the kernel is wrong (see the red box above) or pip install didn't finish.

### Loading your secret key

Your API key lives in a file called `.env`, not in this notebook. That's on purpose — if the
key were written in a cell, then every time you shared this notebook you'd be giving away
your key, and whoever received it could spend your money.

`load_dotenv()` reads that file and loads what's inside into the program's environment,
where `os.getenv` can find it.

The `override=True` part means "if this key was already loaded earlier, load it again
anyway". That matters when you edit `.env` and re-run — without it, Python would keep
using the old value and you'd swear the file hadn't changed.

In [ ]:
# Load the .env file and check the key looks sensible

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ No API key found. Is there a file called exactly '.env' next to this notebook?")
elif not api_key.startswith("sk-"):
    print("❌ A key was found, but it doesn't start with 'sk-'. Did you paste the whole thing?")
elif api_key.strip() != api_key:
    print("❌ A key was found, but it has a space or a tab at the start or end. Delete it.")
else:
    print(f"✅ API key found, and it begins {api_key[:8]}... - looks good.")

Notice what that cell did: it didn't just fail or succeed, it told you **which** of four
different things went wrong. Writing checks like that for yourself is a habit worth
stealing. A program that says "something went wrong" is useless. A program that says
"your key has a space at the end" has just saved you twenty minutes.

## PART B: Say hello to a model

Before we ask for a picture, let's ask for words. Words are cheaper, faster, and if
something is broken it's much easier to see.

In [ ]:
# These are our settings for today. Capital letters is a Python convention meaning
# "this is a constant - it's set once and not changed while the program runs".

CHAT_MODEL = "openai:gpt-4.1-mini"

### What is that string?

`"openai:gpt-4.1-mini"` is two things joined by a colon:

- `openai` — **which company's** model
- `gpt-4.1-mini` — **which model** of theirs

We hand that string to LangChain's `init_chat_model`, and it gives us back an object we can
talk to. The important part, and the reason we're doing it this way instead of using
OpenAI's own library directly: **later in this course we will change that string to a
different company, and nothing else in our code will have to change.**

In [ ]:
# Create the model. This doesn't call anything over the internet yet -
# it just builds the object that knows how to make calls.

model = init_chat_model(CHAT_MODEL)
model

That cell has no `print()` in it. In a notebook, if the last line of a cell is just a value,
the notebook shows it to you. We'll use that constantly to poke at things and see what
they are. Get into the habit of typing a variable's name on its own line to look inside it.

In [ ]:
# Our first actual call over the internet. This costs a fraction of a cent.

response = model.invoke("In one short sentence, tell me something surprising about the colour blue.")
response.text

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🔄 Coming from JavaScript?</h3>
<p style="color:#06c; margin-bottom:0;">
<code>model.invoke(...)</code> is a <b>method call</b> — exactly like <code>array.map(...)</code>
in JavaScript. Object, dot, name, parentheses. Same idea, same punctuation.<br/><br/>
And notice there's no <code>await</code>. In Python you <i>can</i> write asynchronous code,
but you don't have to, and we won't. <code>invoke</code> just waits for the answer and
hands it to you. This is one of the places Python is genuinely simpler than JavaScript.
</p>
</div>

In [ ]:
# Run the same cell again. You'll get a different sentence.
# These models are not looking anything up - they're generating, fresh, every time.

model.invoke("In one short sentence, tell me something surprising about the colour blue.").text

## PART C: Your first image

Now the part you came for.

An image model is reached the same way — you `invoke` it with a description. The one
difference is that we have to tell the model it's *allowed* to draw, by handing it a tool.

In [ ]:
# Ask the model for the ability to generate images.
# "quality": "low" is faster and much cheaper - it's the right setting while experimenting.
# We'll turn it up at the very end of the course for our final showcase images.

IMAGE_TOOL = {"type": "image_generation", "quality": "low"}

image_model = init_chat_model(CHAT_MODEL).bind_tools([IMAGE_TOOL])

`bind_tools` returns a **new** model object that has the drawing tool attached. The original
`model` is untouched. That's a pattern you'll see a lot: instead of changing something,
you make a new version of it with the change applied.

In [ ]:
# Here is our prompt. Change this to anything you like - that's the whole point.

prompt = "A red bicycle leaning against a wall covered in ivy, warm afternoon light"

In [ ]:
# This is the slow one. Expect to wait 10-30 seconds. That is normal, not a crash.

result = image_model.invoke(prompt)
print("Done!")

### What did we actually get back?

Not a picture. We got a message, and that message is made of **content blocks** — a list of
pieces. Some pieces are text, some are images. Let's look at what kinds we got:

In [ ]:
# Look at what types of block came back

for block in result.content_blocks:
    print(block["type"])

In [ ]:
# Now look at one whole block - but only the first 300 characters of it,
# because an image block is enormous and would flood the screen.

str(result.content_blocks[-1])[:300]

### Base64: why a picture arrives as text

You just saw the image arrive as a gigantic string of letters and numbers.

APIs like this one send **text**, not files. So to send you a picture, it has to be
converted into text first — and the standard way of doing that is called **base64**. It
takes the raw bytes of a file and re-writes them using only ordinary characters that survive
being sent over the internet.

Our job is to do that in reverse: find the image block, take its base64 text, and **decode**
it back into bytes.

In [ ]:
# Find the first block that is an image.
# The `if block["type"] == "image"` part is a filter - it skips the text blocks.

image_blocks = [block for block in result.content_blocks if block["type"] == "image"]

print(f"Found {len(image_blocks)} image(s)")

In [ ]:
# Take the base64 text out of the first image block, and decode it into real bytes

image_base64 = image_blocks[0]["base64"]
image_bytes = base64.b64decode(image_base64)

print(f"The base64 text is {len(image_base64):,} characters long")
print(f"The decoded image is  {len(image_bytes):,} bytes")

In [ ]:
# And finally: show it.

display(Image(image_bytes))

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
That exact chain — <b>send text → receive base64 → decode to bytes → show or save</b> —
is how every "generate an image" button on every website works, from Canva to Photoshop's
generative fill. You have just written the actual thing, not a toy version of it. The
companies have nicer buttons; the middle of the pipeline is what you just did.
</p>
</div>

## PART D: Save it, then break it on purpose

In [ ]:
# Write the bytes straight to a file. "wb" means "write, binary" -
# binary because this is a picture, not text.

Path("my_first_image.png").write_bytes(image_bytes)

print("Saved! Look in the file list on the left of VS Code.")

### Now break it

Seriously. You learn far more from a broken program than a working one, and it is much
less frightening to break something on purpose than to have it break at you.

Run the cell below. Read the error. Then read the last line of the error first — Python
puts the actual problem at the **bottom**, and everything above it is just the trail of how
it got there.

In [ ]:
# This will fail. That's the point. Read the last line.

base64.b64decode("this is definitely not base64 🙂")

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Go back up to the cell that sets <code>prompt</code> and write your own description.
Then re-run the cells from there down.<br/><br/>
Make <b>three</b> images. Try to make them very different from each other.<br/><br/>
Then run the <i>same</i> prompt twice without changing a single character, and look
carefully at the two results. What do you notice?
</p>
</div>

In [ ]:
# Your turn. Change the prompt, run this cell, see what you get.

prompt = "your description goes here"

result = image_model.invoke(prompt)
image_bytes = base64.b64decode(
    [b for b in result.content_blocks if b["type"] == "image"][0]["base64"]
)
display(Image(image_bytes))

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Finished early? Try describing something the model has almost certainly never seen a
photograph of — "a giraffe operating a submarine periscope", "a library made entirely of
ice". Where does it do well? Where does it get confused and produce something that is
almost right but anatomically impossible? Keep the funniest failure; we'll talk about why
it happens.
</p>
</div>

## Where we got to

In one session you have:

- a working local Python environment, which is genuinely the hardest part of any first day
- a secret key stored safely, outside your code
- a model created through **one string** you'll later change to switch companies
- a real API call, and an understanding that base64 → bytes → file is the whole pipeline
- an image you described, saved on your own computer

## Next session

That last cell was ugly. Six lines of unreadable brackets every single time we want one
picture. Next session we learn enough Python to turn it into this:

```python
image = make_image("a red bicycle covered in ivy")
```

...and you'll spend the rest of the course calling that instead.